Note, to run this notebook you need to slightly change the format for starting a coiled session:
> `AWS_ACCESS_KEY_ID="" AWS_SECRET_ACCESS_KEY="" AWS_SESSION_TOKEN="" AWS_PROFILE="" uv run coiled notebook start --vm-type r8g.8xlarge --sync --sync-ignore .venv`

In [1]:
import icechunk

from srm.config import _icechunk_storage_for_path
from srm.qa_flags import (
    ATTRS_TIME_INVARIANT,
    ATTRS_TIME_VARYING,
    FLAG_LIST_TIME_INVARIANT,
    FLAG_LIST_TIME_VARYING,
    combine_intermediate_flags,
    discover_leaves,
    parse_tag,
    write_final_qa_flags,
)

In [2]:
flag_time_invariant_name = ATTRS_TIME_INVARIANT["short_name"]
flag_time_varying_name = ATTRS_TIME_VARYING["short_name"]

In [3]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]

In [4]:
GCMS = ["CESM2-WACCM6", "UKESM1-1-LL"]

BRANCH = "v0.14.1"
ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
STORE_SUBSET_ID = "global"

PLOT_FLAG_MAPS = True
BUCKET = "carbonplan-srm"
PREFIX = "scratch/output/qa-intermediate-flags-v0.14.1"

In [6]:
import coiled
from frisky import hijack

cluster = coiled.Cluster(
    name="srm-qaqc-flags-step3",
    region="us-west-2",
    n_workers=12,
    worker_vm_types=["m8gn.xlarge"],
    scheduler_vm_types="c8g.xlarge",
    spot_policy="spot_with_fallback",
    use_best_zone=True,
    tags={"Project": "SRM"},
    worker_options={"nthreads": 8},
    environ={"ZARR_ASYNC__CONCURRENCY": "128"},
)

client = hijack(cluster.get_client())
client

[2026-09-07 11:57:52,919][INFO    ][coiled] Fetching latest package priorities...
[2026-09-07 11:57:52,919][INFO    ][coiled.package_sync] Resolving your local /Users/clairezarakas/Documents/science/srm-downscaling/uv.lock Python environment...
[2026-09-07 11:57:53,142][INFO    ][coiled.package_sync] Scanning 293 python packages...
[2026-09-07 11:57:53,471][INFO    ][coiled] Running pip check...
[2026-09-07 11:57:53,857][INFO    ][coiled] Validating environment...
[2026-09-07 11:57:54,658][INFO    ][coiled] Creating wheel for ~/Documents/science/srm-downscaling/src...
[2026-09-07 11:57:54,762][INFO    ][coiled] Creating wheel for srm...
[2026-09-07 11:57:56,471][INFO    ][coiled] Uploading coiled_local_src...
[2026-09-07 11:57:57,477][INFO    ][coiled] Uploading srm...
[2026-09-07 11:57:58,485][INFO    ][coiled] Creating software environment...
[2026-09-07 11:58:01,422][INFO    ][coiled] Creating Cluster (name: srm-qaqc-flags-step3, https://cloud.coiled.io/clusters/2018153 ). This usua

<frisky.Client: scheduler="wss://cluster-mjufr.dask.host/DQ_9EQMCCkUtgH5h/frisky-comm?__frisky_dial_host=54.201.159.213" id="client-0">

In [6]:
import logging

logging.basicConfig(level=logging.INFO)

# Option 1: call `run_step3` function (goes through all steps at once, for both downscaled and debiased coarse)

In [7]:
from srm.qa_flags import run_step3

run_step3(
    gcms=GCMS,
    branch=BRANCH,
    root_dir=ROOT_DIR,
    store_subset_id=STORE_SUBSET_ID,
    bucket=BUCKET,
    prefix=PREFIX,
    overwrite=False,
    verbose=True,
    mode="both",
)

INFO:srm.qa_flags:Discovering leaves of the data tree...
INFO:srm.qa_flags:opened 2/2 stores on branch 'v0.14.1': CESM2-WACCM6, UKESM1-1-LL
INFO:srm.qa_flags:200 leaves across 2 GCMs
INFO:srm.qa_flags:  leaf discovery completed in 46.0s
INFO:srm.qa_flags:Writing final flags for downscaled data...
INFO:srm.qa_flags:92 tags to process
INFO:srm.qa_flags:CESM2-WACCM6_pr_g6_1p5k_002_bcsd
INFO:srm.qa_flags:  commit 5015ZVCP4CR69E978B70
INFO:srm.qa_flags:CESM2-WACCM6_pr_g6_1p5k_003_bcsd
INFO:srm.qa_flags:  commit BTX8GQG4WSXC0TJ99610
INFO:srm.qa_flags:CESM2-WACCM6_rsds_g6_1p5k_002_bcsd
INFO:srm.qa_flags:  commit W2GGR21GJ1DBT6AN5HH0
INFO:srm.qa_flags:CESM2-WACCM6_rsds_g6_1p5k_003_bcsd
INFO:srm.qa_flags:  commit 52W9K26B1MV91E10H5A0
INFO:srm.qa_flags:CESM2-WACCM6_tas_g6_1p5k_002_bcsd
INFO:srm.qa_flags:  commit K076PK7Y0HZ0CY5B0EQ0
INFO:srm.qa_flags:CESM2-WACCM6_tas_g6_1p5k_003_bcsd
INFO:srm.qa_flags:  commit EGGCRAEKK6HA340JSRSG
INFO:srm.qa_flags:CESM2-WACCM6_tasmax_g6_1p5k_002_bcsd
INFO:srm.q

# Option 2: go through each step one at a time (just for downscaled, not including debiased coarse in this demo)

### A. Define what data arrays exist to traverse

In [ ]:
[_, tags, _, _, _, tags_np, _, _] = discover_leaves(
    gcms=GCMS, branch=BRANCH, root_dir=ROOT_DIR, store_subset_id=STORE_SUBSET_ID, is_downscaled=True
)

### B. Write out final overall flags

In [ ]:
tag = "CESM2-WACCM6_rsds_ssp245_003_qdmsd"

[overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
    tag=tag,
    flag_list_time_varying=FLAG_LIST_TIME_VARYING,
    flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
    bucket=BUCKET,
    prefix=PREFIX,
)

lat = overall_flag_time_invariant.lat
lon = overall_flag_time_invariant.lon

In [ ]:
print(len(tags))
# This loop takes about 15 minutes to run on v0.13.0 (31 global data arrays)
OVERWRITE = False

repos: dict[str, icechunk.Repository] = {}
for gcm in GCMS:
    repos[gcm] = icechunk.Repository.open(
        _icechunk_storage_for_path(f"{ROOT_DIR}{gcm}-ERA5-{STORE_SUBSET_ID}.icechunk")
    )

for i, tag in enumerate(tags):
    print(tag)
    [gcm, var, scenario, ens, method] = parse_tag(tag)
    print("calculating flags")
    [overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
        tag=tag,
        flag_list_time_varying=FLAG_LIST_TIME_VARYING,
        flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
        bucket=BUCKET,
        prefix=PREFIX,
    )

    group = f"{method}/{scenario}/{var}/{ens}"
    session = repos[gcm].writable_session(BRANCH)

    print("  writing flag_time_varying")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_varying,
        flag_name=flag_time_varying_name,
        attrs=ATTRS_TIME_VARYING,
        overwrite=OVERWRITE,
    )

    print("  writing flag_time_invariant")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_invariant,
        flag_name=flag_time_invariant_name,
        attrs=ATTRS_TIME_INVARIANT,
        overwrite=OVERWRITE,
    )

    commit = session.commit(f"write qa flags for {tag}")
    print(f"    commit {commit}")

In [ ]:
if cluster is not None:
    cluster.shutdown()
else:
    print("no cluster was created (cached run); nothing to shut down")